In [ ]:
import os
import sys
import pandas as pd
import json
import ast
import numpy as np

In [166]:
pd.options.display.max_rows = None
pd.options.display.max_colwidth = None
pd.options.display.max_columns = None

In [507]:
meta_data = pd.read_csv('step_2.csv')

In [ ]:
# relation 처리 필요한 필드 리스트 
sorted(['optionData.dataOptionProvidingMethod', 'productRelation.signupPreTermination.productInformation.productList', 'productRelation.signupConcurrentTermination.productInformation.productList', 'productRelation.signupPreTermination.productInformation.groupList', 'productRelation.signupConcurrentTermination.productInformation.groupList', 'productRelation.terminationPreTermination.productInformation.productList', 'productRelation.terminationConcurrentTermination.productInformation.productList', 'productRelation.terminationConcurrentTermination.productInformation.groupList', 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList', 'productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits', 'otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList', 'productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList', 'productBenefitConditions.allOfferBenefits.benefitInfo'])

['optionData.dataOptionProvidingMethod',
 'otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList',
 'productBenefitConditions.allOfferBenefits.benefitInfo',
 'productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList',
 'productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits',
 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList',
 'productRelation.signupConcurrentTermination.productInformation.groupList',
 'productRelation.signupConcurrentTermination.productInformation.productList',
 'productRelation.signupPreTermination.productInformation.groupList',
 'productRelation.signupPreTermination.productInformation.productList',
 'productRelation.terminationConcurrentTermination.productInformation.groupList',
 'productRelation.terminationConcurrentTermination.productInformation.productList',
 'productRelation.terminationPreTermination.productInformation.productList']

### 2-8) 관계형 : 공통 그룹 메타(optionData)
- 'optionData.dataOptionProvidingMethod'
- 같은 그룹에 속하면 모든 케이스에 동일한 값이 적혀 있어 그룹명 단위로 별도 테이블 생성

In [518]:
tmp_df = pd.DataFrame(meta_data['otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList'].value_counts())
tmp_df.reset_index(inplace=True)
tmp_df.columns = ['group_list', 'count']
tmp_df['group_list'] = tmp_df['group_list'].apply(ast.literal_eval)
tmp_df = tmp_df.explode(column='group_list')
tmp_df.drop(columns='count', inplace=True)
tmp_df = tmp_df.drop_duplicates()
tmp_df['groupName'] = tmp_df.apply(lambda x: x['group_list'].get('groupName'), axis = 1)
tmp_df['groupList'] = tmp_df['group_list'].apply(lambda x: x.get('groupList'))
tmp_df.drop(columns = 'group_list', inplace = True)

In [ ]:
product_group = tmp_df.explode(column='groupList')
product_group = product_group.reset_index(drop=True)
product_group['pmProductId'] =  product_group.apply(lambda x: x['groupList'].get('pmProductId'), axis = 1)
product_group['legacyProductId'] =  product_group.apply(lambda x: x['groupList'].get('legacyProductId'), axis = 1)
product_group['productName'] =  product_group.apply(lambda x: x['groupList'].get('productName'), axis = 1)
product_group.drop(columns='groupList', inplace=True)
product_group.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/product_group.csv',index=False)

In [523]:
# 전체 meta_data에는 groupName만 남김
result_list = []
for i in range(len(meta_data)):
    value = meta_data['otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList'][i]
    try:
        value = ast.literal_eval(value)
        value = [item.get('groupName') for item in value]
        result_list.append(value)
    except:
        result_list.append(None)


In [524]:
meta_data['otherOnboardInfo.duplicateNameOnboard.productGroup.groupList'] = result_list
meta_data.drop(columns='otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList', inplace=True)

In [525]:
meta_data.to_csv('step_3.csv', index=False)

### 2-9) 관계형 : ProductBenefit 컬럼 3종을 병합하여 하나로 생성, 제공되는 혜택 상품코드/명만 추출하여 relation_db로 생성
- productBenefitConditions.allOfferBenefits.benefitInfo, 
- productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList, 
- productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList

In [ ]:
# productBenefitConditions.allOfferBenefits.benefitInfo, productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList, productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList
# 병합해서 하나로 만들기 
   -> productBenefitConditions.allBenefitList 

In [526]:
result_list = []
for i in range(len(meta_data)):
    val_a = meta_data['productBenefitConditions.allOfferBenefits.benefitInfo'][i]
    val_b = meta_data['productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList'][i]
    val_c = meta_data['productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList'][i]
    try:
        val_a = ast.literal_eval(val_a)
    except:
        val_a = []
    try:
        val_b = ast.literal_eval(val_b)
    except:
        val_b = []
    try:
        val_c = ast.literal_eval(val_c)
    except:
        val_c = []
    val_merged = val_a + val_b + val_c
    seen = set()
    seen_list = []
    for d in val_merged:
        t = tuple(sorted(d.items()))
        if t not in seen:
            seen.add(t)
            seen_list.append(d)
    result_list.append(seen_list)

In [527]:
len(result_list)

125

In [528]:
# meta_data에 병합된 결과 삽입
meta_data['productBenefitConditions.allBenefitList'] = result_list
meta_data.drop(columns = ['productBenefitConditions.allOfferBenefits.benefitInfo','productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList','productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList' ], inplace=True)

In [529]:
meta_data[['productBenefitConditions.allBenefitList']].head(1)

,productBenefitConditions.allBenefitList
0,"[{'productId': 'BA00000047', 'productName': 'Wavve 2천원 할인'}]"


In [ ]:
# 별도 relation_db로 추출 
    # 'productBenefitConditions.allOfferBenefits.benefitInfo',
    # 'productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList',
    # 'productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits',
    # 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList',
tmp_df = meta_data[['pmProductID','productBenefitConditions.allBenefitList']]
tmp_df = tmp_df.explode(column = 'productBenefitConditions.allBenefitList')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df = tmp_df[~tmp_df['productBenefitConditions.allBenefitList'].isnull()]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['productId'] = tmp_df.apply(lambda x: list(x['productBenefitConditions.allBenefitList'].values())[0], axis = 1)
tmp_df['productName'] = tmp_df.apply(lambda x: list(x['productBenefitConditions.allBenefitList'].values())[1], axis = 1)
tmp_df['type'] = 'productBenefitConditions.allBenefitList'
tmp_df.drop(columns= ['productBenefitConditions.allBenefitList'],inplace=True)

In [ ]:
relation_db = pd.DataFrame()
relation_db = pd.concat([relation_db, tmp_df])

In [274]:
relation_db.head(1)

,pmProductID,productId,productName,type
0,PA00000001,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList


### 2-10) 관계형 : optionData
- optionData.dataOptionProvidingMethod
- 월요금 정보 등은 제외 

In [ ]:
tmp_df = meta_data[['pmProductID', 'optionData.dataOptionProvidingMethod']]
tmp_df['optionData.dataOptionProvidingMethod'] = tmp_df['optionData.dataOptionProvidingMethod'].apply(ast.literal_eval)
tmp_df = tmp_df.explode(column = 'optionData.dataOptionProvidingMethod')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['productId'] = tmp_df['optionData.dataOptionProvidingMethod'].apply(lambda x: x.get('productId'))
tmp_df['productName'] = tmp_df['optionData.dataOptionProvidingMethod'].apply(lambda x: x.get('productName'))
tmp_df = tmp_df[~(tmp_df['productId'].isnull()&tmp_df['productName'].isnull())]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['type'] = 'optionData.dataOptionProvidingMethod'
tmp_df.drop(columns=['optionData.dataOptionProvidingMethod'],inplace=True)
tmp_df.head(5)

,pmProductID,productId,productName,type
0,PA00000002,None,심야데이터 75% 할인,optionData.dataOptionProvidingMethod
1,PA00000003,None,심야데이터 75% 할인,optionData.dataOptionProvidingMethod
2,PA00000011,None,심야데이터 75% 할인,optionData.dataOptionProvidingMethod
3,PA00000028,-,매일 2GB+3Mbps 데이터 추가,optionData.dataOptionProvidingMethod
4,PA00000030,-,매일 2GB+3Mbps 데이터 추가,optionData.dataOptionProvidingMethod


In [ ]:
relation_db = pd.concat([relation_db, tmp_df])
relation_db = relation_db.reset_index(drop=True)

In [279]:
relation_db.shape

(376, 4)

### 2-11) 관계형 : productRelation
- groupList와 productList가 의미상은 동일한 것이므로 병합 
- 실제로는 두 컬럼이 중첩되는 값 없이 별도의 값을 가지고 있음 

 'productRelation.signupConcurrentTermination.productInformation.groupList',
 'productRelation.signupConcurrentTermination.productInformation.productList',
 'productRelation.signupPreTermination.productInformation.groupList',
 'productRelation.signupPreTermination.productInformation.productList',
 'productRelation.terminationConcurrentTermination.productInformation.groupList',
 'productRelation.terminationConcurrentTermination.productInformation.productList',
 'productRelation.terminationPreTermination.productInformation.groupList'
 'productRelation.terminationPreTermination.productInformation.productList'

In [530]:
meta_data['productRelation.signupConcurrentTermination.productInformation.groupList'] = meta_data['productRelation.signupConcurrentTermination.productInformation.groupList'].fillna('[]')
meta_data['productRelation.signupConcurrentTermination.productInformation.groupList'] = meta_data['productRelation.signupConcurrentTermination.productInformation.groupList'].apply(ast.literal_eval)
meta_data['productRelation.signupConcurrentTermination.productInformation.productList'] = meta_data['productRelation.signupConcurrentTermination.productInformation.productList'].fillna('[]')
meta_data['productRelation.signupConcurrentTermination.productInformation.productList'] = meta_data['productRelation.signupConcurrentTermination.productInformation.productList'].apply(ast.literal_eval)

In [531]:
def merged_list(col_a, col_b):
    list_a = []
    for i in range(len(meta_data)):
        value = meta_data[col_a][i]
        if len(value)==0:
            list_a.append([])
        else:
            list_a.append(value[0]['groupList'])

    list_b = []
    for i in range(len(meta_data)):
        value = meta_data[col_b][i]
        if len(value)==0:
            list_b.append([])
        else:
            list_b.append(value)

    result_list = [a+b for a, b in zip(list_a, list_b)]
    return result_list

In [532]:
meta_data['productRelation.signupConcurrentTermination.productList'] = merged_list('productRelation.signupConcurrentTermination.productInformation.groupList', 'productRelation.signupConcurrentTermination.productInformation.productList')

In [533]:
meta_data['productRelation.signupPreTermination.productInformation.groupList'] = meta_data['productRelation.signupPreTermination.productInformation.groupList'].fillna('[]')
meta_data['productRelation.signupPreTermination.productInformation.groupList'] = meta_data['productRelation.signupPreTermination.productInformation.groupList'].apply(ast.literal_eval)
meta_data['productRelation.signupPreTermination.productInformation.productList'] = meta_data['productRelation.signupPreTermination.productInformation.productList'].fillna('[]')
meta_data['productRelation.signupPreTermination.productInformation.productList'] = meta_data['productRelation.signupPreTermination.productInformation.productList'].apply(ast.literal_eval)

In [534]:
meta_data['productRelation.signupPreTermination.productList'] = merged_list('productRelation.signupPreTermination.productInformation.groupList', 'productRelation.signupPreTermination.productInformation.productList')

In [535]:
meta_data['productRelation.terminationConcurrentTermination.productInformation.groupList'] = meta_data['productRelation.terminationConcurrentTermination.productInformation.groupList'].fillna('[]')
meta_data['productRelation.terminationConcurrentTermination.productInformation.groupList'] = meta_data['productRelation.terminationConcurrentTermination.productInformation.groupList'].apply(ast.literal_eval)
meta_data['productRelation.terminationConcurrentTermination.productInformation.productList'] = meta_data['productRelation.terminationConcurrentTermination.productInformation.productList'].fillna('[]')
meta_data['productRelation.terminationConcurrentTermination.productInformation.productList'] = meta_data['productRelation.terminationConcurrentTermination.productInformation.productList'].apply(ast.literal_eval)

In [536]:
meta_data['productRelation.terminationConcurrentTermination.productList'] = merged_list('productRelation.terminationConcurrentTermination.productInformation.groupList', 'productRelation.terminationConcurrentTermination.productInformation.productList')

In [537]:
meta_data['productRelation.terminationPreTermination.productInformation.groupList'] = meta_data['productRelation.terminationPreTermination.productInformation.groupList'].fillna('[]')
meta_data['productRelation.terminationPreTermination.productInformation.groupList'] = meta_data['productRelation.terminationPreTermination.productInformation.groupList'].apply(ast.literal_eval)
meta_data['productRelation.terminationPreTermination.productInformation.productList'] = meta_data['productRelation.terminationPreTermination.productInformation.productList'].fillna('[]')
meta_data['productRelation.terminationPreTermination.productInformation.productList'] = meta_data['productRelation.terminationPreTermination.productInformation.productList'].apply(ast.literal_eval)

In [538]:
meta_data['productRelation.terminationPreTermination.productList'] = merged_list('productRelation.terminationPreTermination.productInformation.groupList', 'productRelation.terminationPreTermination.productInformation.productList')

In [539]:
meta_data.drop(columns = [ 'productRelation.signupConcurrentTermination.productInformation.groupList',
 'productRelation.signupConcurrentTermination.productInformation.productList',
 'productRelation.signupPreTermination.productInformation.groupList',
 'productRelation.signupPreTermination.productInformation.productList',
 'productRelation.terminationConcurrentTermination.productInformation.groupList',
 'productRelation.terminationConcurrentTermination.productInformation.productList',
 'productRelation.terminationPreTermination.productInformation.groupList',
 'productRelation.terminationPreTermination.productInformation.productList'], inplace=True)

In [540]:
meta_data.to_csv('step_3.csv', index=False)

In [ ]:
# relation_db에 반영 


In [401]:
# productRelation.signupConcurrentTermination.productList
tmp_df = meta_data[['pmProductID','productRelation.signupConcurrentTermination.productList']]
tmp_df = tmp_df.explode(column = 'productRelation.signupConcurrentTermination.productList')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df = tmp_df[~tmp_df['productRelation.signupConcurrentTermination.productList'].isnull()]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['pmProductId'] = tmp_df.apply(lambda x: x['productRelation.signupConcurrentTermination.productList']['pmProductId'], axis = 1)
tmp_df['productName'] = tmp_df.apply(lambda x: x['productRelation.signupConcurrentTermination.productList']['productName'], axis = 1)
tmp_df.columns = ['pmProductID', 'productRelation.signupConcurrentTermination.productList', 'productId', 'productName']
tmp_df['type'] = 'productRelation.signupConcurrentTermination.productList'
tmp_df.drop(columns = 'productRelation.signupConcurrentTermination.productList', inplace=True)

In [402]:
relation_db = pd.concat([relation_db, tmp_df])
relation_db.shape

(2962, 4)

In [403]:
# productRelation.signupPreTermination.productList
tmp_df = meta_data[['pmProductID','productRelation.signupPreTermination.productList']]
tmp_df = tmp_df.explode(column = 'productRelation.signupPreTermination.productList')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df = tmp_df[~tmp_df['productRelation.signupPreTermination.productList'].isnull()]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['pmProductId'] = tmp_df.apply(lambda x: x['productRelation.signupPreTermination.productList']['pmProductId'], axis = 1)
tmp_df['productName'] = tmp_df.apply(lambda x: x['productRelation.signupPreTermination.productList']['productName'], axis = 1)
tmp_df.columns = ['pmProductID', 'productRelation.signupPreTermination.productList', 'productId', 'productName']
tmp_df['type'] = 'productRelation.signupPreTermination.productList'
tmp_df.drop(columns = 'productRelation.signupPreTermination.productList', inplace=True)

In [404]:
relation_db = pd.concat([relation_db, tmp_df])
relation_db.shape

(4156, 4)

In [405]:
# productRelation.terminationConcurrentTermination.productList
tmp_df = meta_data[['pmProductID','productRelation.terminationConcurrentTermination.productList']]
tmp_df = tmp_df.explode(column = 'productRelation.terminationConcurrentTermination.productList')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df = tmp_df[~tmp_df['productRelation.terminationConcurrentTermination.productList'].isnull()]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['pmProductId'] = tmp_df.apply(lambda x: x['productRelation.terminationConcurrentTermination.productList']['pmProductId'], axis = 1)
tmp_df['productName'] = tmp_df.apply(lambda x: x['productRelation.terminationConcurrentTermination.productList']['productName'], axis = 1)
tmp_df.columns = ['pmProductID', 'productRelation.terminationConcurrentTermination.productList', 'productId', 'productName']
tmp_df['type'] = 'productRelation.terminationConcurrentTermination.productList'
tmp_df.drop(columns = 'productRelation.terminationConcurrentTermination.productList', inplace=True)

In [407]:
relation_db = pd.concat([relation_db, tmp_df])
relation_db.shape

(5068, 4)

In [408]:
# productRelation.terminationPreTermination.productList
tmp_df = meta_data[['pmProductID','productRelation.terminationPreTermination.productList']]
tmp_df = tmp_df.explode(column = 'productRelation.terminationPreTermination.productList')
tmp_df = tmp_df.reset_index(drop=True)
tmp_df = tmp_df[~tmp_df['productRelation.terminationPreTermination.productList'].isnull()]
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['pmProductId'] = tmp_df.apply(lambda x: x['productRelation.terminationPreTermination.productList']['pmProductId'], axis = 1)
tmp_df['productName'] = tmp_df.apply(lambda x: x['productRelation.terminationPreTermination.productList']['productName'], axis = 1)
tmp_df.columns = ['pmProductID', 'productRelation.terminationPreTermination.productList', 'productId', 'productName']
tmp_df['type'] = 'productRelation.terminationPreTermination.productList'
tmp_df.drop(columns = 'productRelation.terminationPreTermination.productList', inplace=True)

In [409]:
relation_db = pd.concat([relation_db, tmp_df])
relation_db.shape

(5366, 4)

In [411]:
relation_db['type'].value_counts()

type
productRelation.signupConcurrentTermination.productList         2586
productRelation.signupPreTermination.productList                1194
productRelation.terminationConcurrentTermination.productList     912
productBenefitConditions.allBenefitList                          347
productRelation.terminationPreTermination.productList            298
optionData.dataOptionProvidingMethod                              29
Name: count, dtype: int64

In [ ]:
relation_db['productId'] = relation_db['productId'].fillna('')
relation_db['relationID'] = relation_db.apply(lambda x: x['pmProductID']+'_'+x['productId'], axis = 1)
relation_db = relation_db.drop_duplicates()
relation_db = relation_db.reset_index(drop=True)

In [ ]:
# signupConcurrentTermination(동시해지)와 signupPreTermination(선행해지)의 해석의 차이로 중복되어 삽입된 상품이 있어 보임. -> 이 경우 동시해지만 남기고, 선행 해지는 제거 
tmp_df = pd.DataFrame(relation_db.relationID.value_counts())
tmp_df.reset_index(inplace=True)
relation_db[relation_db.relationID.isin(list(tmp_df[tmp_df['count']>=2]['relationID']))].sort_values(by='relationID')

,pmProductID,productId,productName,type,relationID
1453,PA00000053,PB00001176,긴통화무료옵션,productRelation.signupConcurrentTermination.productList,PA00000053_PB00001176
3403,PA00000053,PB00001176,긴통화무료옵션,productRelation.signupPreTermination.productList,PA00000053_PB00001176
1492,PA00000054,PB00001176,긴통화무료옵션,productRelation.signupConcurrentTermination.productList,PA00000054_PB00001176
3426,PA00000054,PB00001176,긴통화무료옵션,productRelation.signupPreTermination.productList,PA00000054_PB00001176
1531,PA00000055,PB00001176,긴통화무료옵션,productRelation.signupConcurrentTermination.productList,PA00000055_PB00001176
3462,PA00000055,PB00001176,긴통화무료옵션,productRelation.signupPreTermination.productList,PA00000055_PB00001176
1570,PA00000056,PB00001176,긴통화무료옵션,productRelation.signupConcurrentTermination.productList,PA00000056_PB00001176
3478,PA00000056,PB00001176,긴통화무료옵션,productRelation.signupPreTermination.productList,PA00000056_PB00001176


In [ ]:
relation_db.columns = ['pmProductID', 'productId', 'productName', 'type', 'relationID']
relation_db = relation_db[['relationID','pmProductID', 'productId', 'productName', 'type']]
relation_db = relation_db[~((relation_db.relationID.isin(list(tmp_df[tmp_df['count']>=2]['relationID'])))&(relation_db['type']=='productRelation.signupPreTermination.productList'))]
relation_db = relation_db.reset_index(drop=True)

In [632]:
relation_db.shape

(5356, 5)

In [631]:
relation_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/relation_db.csv', index=False)

In [608]:
relation_db.head(1)

,relationID,pmProductID,productId,productName,type
0,PA00000001_BA00000047,PA00000001,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList


### 3-1) 최종 분화

In [541]:
target_list = list(meta_data.columns)
len(target_list)

68

In [542]:
# PRODUCT
target_list = [ item for item in target_list if item not in ['managementInfo.productId.value', 'managementInfo.mappedProductCode.productCode.valueList', 'managementInfo.generation.valueList', 'managementInfo.marketingKeyword.valueList','managementInfo.productName.value','managementInfo.productNameInEnglish.value','managementInfo.lineup.value', 'managementInfo.classifiedGroup.value', 'managementInfo.productDescription.value', 'managementInfo.productSubscriptionCondition.value','managementInfo.statusOfOperation.value']]
len(target_list) # 57개로 감소

57

In [543]:
meta_data[meta_data['pmProductID'] != meta_data['managementInfo.productId.value']] # 중복 정보이므로 제거 

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,smsText.includedText.value,smsText.includedTextSeparateSetting.textRange,includedData.value,additionalDataUsage.includedDataForSharingAndTethering.value,additionalDataUsage.includedMVoIP.value,dataQoS.appliedSpeed.value,seniorDataExceedLimit.availableToApply.value,generalDataExceedLimit.availableToApply.value,monthlyPrice.monthlyPrice.value,monthlyPrice.monthlyPriceWithoutVAT.value,monthlyPrice.monthlyPriceWithSelectableInstallment.value,monthlyPrice.billingMethod.value,optionData.dataOptionProvidingMethod,deductibleInfo.deductibilityForDisability.value,benefitOfData.dataOptionRefill.dataRefillAmount.value,benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value,benefitOfData.dataOptionGift.maximumShareAmount.value,benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value,managementInfo.productId.value,managementInfo.statusOfOperation.value,managementInfo.classifiedGroup.value,managementInfo.productName.value,managementInfo.productNameInEnglish.value,managementInfo.lineup.value,managementInfo.marketingKeyword.valueList,managementInfo.generation.valueList,managementInfo.mappedProductCode.productCode.valueList,managementInfo.productDescription.value,managementInfo.productSubscriptionCondition.value,salesInfo.netPrice.value,topupInfo.reChargeAvailability.availability.value,customerInfo.onboardingCustomer.ageRule,otherOnboardInfo.directPlanOnboard.value,otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value,otherOnboardInfo.tsupportFundOnboard.value,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value,productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits,optionData.optionDataName.value,optionData.selectionMethod.value,optionData.totalNumOfOptions.value,optionData.minNumOfOptionSelectable.value,optionData.maxNumOfOptionSelectable.value,deductibleInfo.additionalOfferForDisabilities.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value,productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value,customerInfo.onboardingCustomer.customerTypeRule.eligibility,customerInfo.onboardingCustomer.customerTypeRule.valueList,topupInfo.chargeAmount.minimumChargeAmount.value,topupInfo.chargeAmount.maximumChargeAmount.value,otherOnboardInfo.specialCustomerOnboard.isSoldier.value,otherOnboardInfo.duplicateNameOnboard.productGroup.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList,benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount,benefitOfVoiceCall.performRefill.voiceCallRefillRange.range,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService,otherOnboardInfo.duplicateNameOnboard.productGroup.groupList,productBenefitConditions.allBenefitList,productRelation.signupConcurrentTermination.productList,productRelation.signupPreTermination.productList,productRelation.terminationConcurrentTermination.productList,productRelation.terminationPreTermination.productList


In [544]:
product_db = meta_data[['pmProductID', 'managementInfo.mappedProductCode.productCode.valueList', 'managementInfo.generation.valueList', 'managementInfo.marketingKeyword.valueList','managementInfo.productName.value','managementInfo.productNameInEnglish.value','managementInfo.lineup.value', 'managementInfo.classifiedGroup.value', 'managementInfo.productDescription.value', 'managementInfo.productSubscriptionCondition.value','managementInfo.statusOfOperation.value']]
product_db.columns = ['pmProductID', 'mappedProductCode', 'generation', 'marketingKeyword', 'productName', 'productNameInEnglish', 'lineup', 'classifiedGroup', 'productDescription', 'productSubscriptionCondition', 'statusOfOperation']
product_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/PRODUCT.csv', index=False)

In [ ]:
# PLAN_CHANGE_CONDITION - 일반 상식으로 프롬프트에 넣는 것으로 하고 제거 
meta_data[['commonRule.planChangeCondition.dailyPlanChangeLimit.value', 'commonRule.planChangeCondition.monthlyPlanChangeLimit.value']].drop_duplicates()

,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value
0,5,1


In [545]:
target_list = [ item for item in target_list if item not in ['commonRule.planChangeCondition.dailyPlanChangeLimit.value', 'commonRule.planChangeCondition.monthlyPlanChangeLimit.value']]
len(target_list) # 55개로 감소

55

In [ ]:
# PRICE
target_list = [ item for item in target_list if item not in ['monthlyPrice.monthlyPrice.value',
 'monthlyPrice.monthlyPriceWithoutVAT.value',
 'monthlyPrice.monthlyPriceWithSelectableInstallment.value',
 'monthlyPrice.billingMethod.value','salesInfo.netPrice.value']]
print(len(target_list)) # 50개로 감소
price_db = meta_data[['pmProductID','monthlyPrice.monthlyPrice.value',
 'monthlyPrice.monthlyPriceWithoutVAT.value',
 'monthlyPrice.monthlyPriceWithSelectableInstallment.value',
 'monthlyPrice.billingMethod.value','salesInfo.netPrice.value']]
price_db.columns = ['pmProductID', 'monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 'billingMethod', 'netPrice']
price_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/PRICE.csv', index=False)

50


In [561]:
# VOICE
target_list = [ item for item in target_list if item not in ['voice.includedVoiceCall.value', 'voice.includedVideoOrValueAddedCall.value', 'voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService',
           'benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount', 'benefitOfVoiceCall.performRefill.voiceCallRefillRange.range']]
print(len(target_list))
voice_db = meta_data[['pmProductID', 'voice.includedVoiceCall.value', 'voice.includedVideoOrValueAddedCall.value', 'voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService',
           'benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount', 'benefitOfVoiceCall.performRefill.voiceCallRefillRange.range']]
voice_db.columns = ['pmProductID', 'includedVoiceCall', 'includedVideoOrValueAddedCall', 'includedVoiceCallTospecifiedNumbers', 'refillAmount', 'refillRange']
voice_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/VOICE.csv', index=False)

45


In [564]:
# SMS
target_list = [ item for item in target_list if item not in ['smsText.includedText.value', 'smsText.includedTextSeparateSetting.textRange']]
print(len(target_list))
sms_db = meta_data[['pmProductID', 'smsText.includedText.value', 'smsText.includedTextSeparateSetting.textRange']]
sms_db.columns = ['pmProductID', 'includedText', 'textRange']
sms_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/SMS.csv', index=False)

43


In [585]:
# DATA 
target_list = [ item for item in target_list if item not in ['includedData.value', 'additionalDataUsage.includedDataForSharingAndTethering.value', 'additionalDataUsage.includedMVoIP.value','dataQoS.appliedSpeed.value',
                                                             'seniorDataExceedLimit.availableToApply.value', 'generalDataExceedLimit.availableToApply.value','benefitOfData.dataOptionRefill.dataRefillAmount.value',
                                                             'benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value', 'benefitOfData.dataOptionGift.maximumShareAmount.value', 'benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value']]
print(len(target_list))
data_db = meta_data[['pmProductID', 'includedData.value', 'additionalDataUsage.includedDataForSharingAndTethering.value', 'additionalDataUsage.includedMVoIP.value','dataQoS.appliedSpeed.value',
                                                             'seniorDataExceedLimit.availableToApply.value', 'generalDataExceedLimit.availableToApply.value', 'benefitOfData.dataOptionRefill.dataRefillAmount.value',
                                                             'benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value', 'benefitOfData.dataOptionGift.maximumShareAmount.value', 'benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value']]
data_db.columns = ['pmProductID', 'includedData', 'includedDataForSharingAndTethering', 'includedMVoIP', 'appliedSpeed', 'seniorDataExceedAvailable', 'generalDataExceedAvailable', 'dataRefillAmount', 'dataRefillCouponGiftingAvailability', 'maximumShareAmount', 'dataGiftReceivingAvailability']
data_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/DATA.csv', index=False)

19


In [ ]:
# TOPUP
target_list = [ item for item in target_list if item not in ['topupInfo.reChargeAvailability.availability.value','topupInfo.chargeAmount.minimumChargeAmount.value',
 'topupInfo.chargeAmount.maximumChargeAmount.value']]
print(len(target_list))
topup_db = meta_data[['pmProductID', 'topupInfo.reChargeAvailability.availability.value','topupInfo.chargeAmount.minimumChargeAmount.value',
 'topupInfo.chargeAmount.maximumChargeAmount.value']]
topup_db.columns = ['pmProductID', 'reChargeAvailability', 'minimumChargeAmount', 'maximumChargeAmount']
topup_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/TOPUP.csv', index=False)


34


In [575]:
# CUSTOMERCONDITION
target_list = [ item for item in target_list if item not in ['customerInfo.onboardingCustomer.ageRule','customerInfo.onboardingCustomer.customerTypeRule.eligibility',
 'customerInfo.onboardingCustomer.customerTypeRule.valueList','customerInfo.onboardingCustomer.individualCustomerSubtypeRule.eligibility',
 'customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList','otherOnboardInfo.directPlanOnboard.value','otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value', 
 'otherOnboardInfo.tsupportFundOnboard.value', 'otherOnboardInfo.duplicateNameOnboard.productGroup.eligibility', 'otherOnboardInfo.duplicateNameOnboard.productGroup.groupList',
 'otherOnboardInfo.specialCustomerOnboard.isSoldier.value']]
print(len(target_list))
c_cond_db = meta_data[['pmProductID', 'customerInfo.onboardingCustomer.ageRule','customerInfo.onboardingCustomer.customerTypeRule.eligibility',
 'customerInfo.onboardingCustomer.customerTypeRule.valueList','customerInfo.onboardingCustomer.individualCustomerSubtypeRule.eligibility',
 'customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList','otherOnboardInfo.directPlanOnboard.value','otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value', 
 'otherOnboardInfo.tsupportFundOnboard.value', 'otherOnboardInfo.duplicateNameOnboard.productGroup.eligibility', 'otherOnboardInfo.duplicateNameOnboard.productGroup.groupList',
 'otherOnboardInfo.specialCustomerOnboard.isSoldier.value']]
c_cond_db.columns = ['pmProductID', 'ageRule', 'customerTypeEligibility', 'customerTypeValueList', 'individualCustomerSubtypeEligibility', 'individualCustomerSubtypeValueList',
                     'directPlanOnboard', 'fixedPlanContractConcurrentSignupRestriction', 'tsupportFundOnboard', 'duplicateNameOnboardEligibility', 'duplicateNameOnboardGroupList', 'specialCustomerIsSoldier']
c_cond_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/CUSTOMERCONDITION.csv', index=False)

23


In [590]:
# BENEFITCONDITION 
target_list = [ item for item in target_list if item not in ['productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value',
 'productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value',
 'productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value',
 'productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value']]
print(len(target_list))
b_cond_db = meta_data[['pmProductID', 'productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value',
 'productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value',
 'productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value',
 'productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value']]
b_cond_db.columns = ['pmProductID', 'selectableBenefitCount', 'selectableBenefitCountPeriodFrom', 'selectableBenefitCountPeriodTo', 'isAutoEnrollmentBenefitOnSignup']
b_cond_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/BENEFITCONDITION.csv', index=False)

15


In [620]:
# relation 제외된 target_list 추출 
target_list = [ item for item in target_list if item not in ['optionData.dataOptionProvidingMethod',
 'otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList',
 'productBenefitConditions.allOfferBenefits.benefitInfo',
 'productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList',
 'productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits',
 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList',
 'productRelation.signupConcurrentTermination.productInformation.groupList',
 'productRelation.signupConcurrentTermination.productInformation.productList',
 'productRelation.signupPreTermination.productInformation.groupList',
 'productRelation.signupPreTermination.productInformation.productList',
 'productRelation.terminationConcurrentTermination.productInformation.groupList',
 'productRelation.terminationConcurrentTermination.productInformation.productList',
 'productRelation.terminationPreTermination.productInformation.productList',
 'productBenefitConditions.allBenefitList']]

In [622]:
# productRelation_*류는 이미 relation_db 들어가 있는 내용의 중복이므로 제거 
target_list = [item for item in target_list if not item.startswith('productRelation')]

In [596]:
# relation_db에 있는 optionData에 포함된 내용이라고 보고 제거 
target_list = [ item for item in target_list if item not in ['optionData.optionDataName.value',
 'optionData.selectionMethod.value',
 'optionData.totalNumOfOptions.value',
 'optionData.minNumOfOptionSelectable.value',
 'optionData.maxNumOfOptionSelectable.value']]
meta_data[['optionData.optionDataName.value',
 'optionData.selectionMethod.value',
 'optionData.totalNumOfOptions.value',
 'optionData.minNumOfOptionSelectable.value',
 'optionData.maxNumOfOptionSelectable.value']].head(5)

,optionData.optionDataName.value,optionData.selectionMethod.value,optionData.totalNumOfOptions.value,optionData.minNumOfOptionSelectable.value,optionData.maxNumOfOptionSelectable.value
0,NaN,NaN,NaN,NaN,NaN
1,심야데이터 할인,포함 옵션,1.0,1.0,1.0
2,심야데이터 할인,포함 옵션,1.0,1.0,1.0
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


In [625]:
# DEDUCTIBLE
target_list = [ item for item in target_list if item not in ['deductibleInfo.deductibilityForDisability.value', 'deductibleInfo.additionalOfferForDisabilities.value']]
print(len(target_list))
deduct_db = meta_data[['pmProductID', 'deductibleInfo.deductibilityForDisability.value', 'deductibleInfo.additionalOfferForDisabilities.value']]
deduct_db.columns = ['pmProductID', 'deductibilityForDisability', 'additionalOfferForDisabilities']
deduct_db.to_csv('/Users/hazel/Documents/map-search-agent/notebooks/rdb/DEDUCTIBLE.csv', index=False)

1


In [ ]:
# PK가 되는 ID 말고는 모두 포함됨 확인 
target_list

['pmProductID']